# Visual Layout Detection Comparison
This notebook compares the visual layout detection bounding boxes of the three benchmarked models:
1. **DocLayout-YOLOv10** (Fast local segmenter)
2. **NVIDIA Nemotron-Parse-v1.1** (Generative local layout)
3. **LandingAI ADE DPT-2** (Cloud parsing API)

Overlays are drawn side-by-side for comparison.

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

# Load input page image
pdf_name = 'Scientific_001'
img_path = 'data/scientific/Scientific_001.png'

if not os.path.exists(img_path):
    # Fallback to create a mock blank white page
    img = Image.new('RGB', (1200, 1600), color='white')
    os.makedirs(os.path.dirname(img_path), exist_ok=True)
    img.save(img_path)
else:
    img = Image.open(img_path)

print(f"Loaded image: {img_path} | Dimensions: {img.size}")

In [ ]:
# Let's run layout detection using our new extractors
from algorithms.layout_detection.doclayout_yolo.extractor import detect_layout as yolo_detect
from algorithms.layout_detection.nemotron_parse.extractor import detect_layout as nemotron_detect
from algorithms.layout_detection.landingai_ade.extractor import detect_layout as ade_detect

print("Running DocLayout-YOLO...")
yolo_res = yolo_detect(img)

print("Running NVIDIA Nemotron-Parse...")
nemotron_res = nemotron_detect(img)

print("Running LandingAI ADE DPT-2...")
ade_res = ade_detect(img)

print("Extraction complete!")

In [ ]:
# Draw overlays and display side-by-side
colors = {
    'title': 'red',
    'text': 'blue',
    'table': 'purple',
    'figure': 'green',
    'picture': 'green',
    'section_header': 'pink',
    'caption': 'cyan'
}

def draw_boxes(image, detections):
    canvas = image.copy()
    draw = ImageDraw.Draw(canvas)
    for det in detections:
        box = det['bbox']
        label = det['type']
        color = colors.get(label, 'grey')
        draw.rectangle(box, outline=color, width=3)
        draw.text((box[0] + 5, box[1] + 5), label, fill=color)
    return canvas

yolo_canvas = draw_boxes(img, yolo_res)
nemotron_canvas = draw_boxes(img, nemotron_res)
ade_canvas = draw_boxes(img, ade_res)

fig, axes = plt.subplots(1, 3, figsize=(24, 12))
axes[0].imshow(yolo_canvas)
axes[0].set_title('DocLayout-YOLOv10')
axes[0].axis('off')

axes[1].imshow(nemotron_canvas)
axes[1].set_title('NVIDIA Nemotron-Parse-v1.1')
axes[1].axis('off')

axes[2].imshow(ade_canvas)
axes[2].set_title('LandingAI ADE-DPT2')
axes[2].axis('off')

plt.tight_layout()
plt.show()